# UniSAR teacher: private Colab training

Build and train the independent UniSAR implementation on public KuaiSAR logs.
This is the clean chronological adaptation documented in the repository, not verified numerical reproduction.

**Prerequisites:** GPU runtime, Google Drive space, and GitHub read access to the private package and TRACE repositories.
Run sections 1–5 first. **Teacher, resume and baseline training gates all default to disabled.**
Review preflight, enable the chosen training gate, and run that cell yourself.

Outline: authenticate/install → persistent storage → verified dataset → GPU preflight → manual teacher run/resume → export/validation → optional matched baselines.


## 1. Authenticate, clone pinned sources, install

Select **Runtime → Change runtime type → GPU** first. The hidden token prompt accepts a fine-grained GitHub token with read access to both repositories. It is passed only to a temporary Git askpass helper; it is not embedded in URLs or saved in notebook outputs.
The TRACE checkout supplies optional existing baselines; LFS weight downloads are skipped.


In [ ]:
from pathlib import Path
import getpass
import os
import subprocess
import sys
import tempfile

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError("Use an available Colab runtime with Python 3.11–3.13 for the pinned preparation dependencies.")

PACKAGE_REVISION = "2090f47f728e1445e4427cce9d9213ce73b4e576"
TRACE_REVISION = "e93ab62af8491acdfc82dfd3054c17f329307d18"
PACKAGE_ROOT = Path("/content/unisar-pytorch")
TRACE_ROOT = Path("/content/TRACE")

def clone_private(repository, destination, revision, token):
    if len(revision) != 40 or not all(c in "0123456789abcdef" for c in revision):
        raise ValueError("Notebook requires an immutable source revision; use the delivered notebook.")
    if not destination.exists():
        with tempfile.TemporaryDirectory(prefix="unisar-auth-") as directory:
            askpass = Path(directory) / "askpass.sh"
            lines = ['#!/bin/sh', 'case "$1" in',
                     '*Username*) printf "%s" "x-access-token";;',
                     '*) printf "%s" "$UNISAR_GH_TOKEN";;', 'esac', '']
            askpass.write_text(chr(10).join(lines), encoding="utf-8")
            askpass.chmod(0o700)
            environment = dict(os.environ, UNISAR_GH_TOKEN=token, GIT_ASKPASS=str(askpass),
                               GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
            result = subprocess.run(["git", "-c", "credential.helper=", "clone", "--quiet",
                                     f"https://github.com/{repository}.git", str(destination)],
                                    env=environment, capture_output=True, text=True)
            if result.returncode:
                raise RuntimeError(f"Private clone failed for {repository}. Check token repository access.")
        subprocess.run(["git", "-C", str(destination), "checkout", "--quiet", "--detach", revision],
                       check=True, env=dict(os.environ, GIT_LFS_SKIP_SMUDGE="1"))
    actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()
    if actual != revision:
        raise RuntimeError(f"Existing {destination} uses another revision. Use a fresh runtime or a new checkout path.")
    changes = subprocess.check_output(["git", "-C", str(destination), "status", "--porcelain",
                                       "--untracked-files=all", "--", "unisar", "src", "scripts",
                                       "configs", "pyproject.toml", "requirements-data.txt"], text=True)
    if changes.strip():
        raise RuntimeError("Source checkout has modifications; use an unchanged pinned source checkout.")
    return actual

token = getpass.getpass("GitHub token (hidden): ") if not (PACKAGE_ROOT.exists() and TRACE_ROOT.exists()) else ""
try:
    package_sha = clone_private("Cyaside/unisar-pytorch", PACKAGE_ROOT, PACKAGE_REVISION, token)
    baseline_sha = clone_private("Cyaside/TRACE", TRACE_ROOT, TRACE_REVISION, token)
finally:
    del token
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r",
                str(PACKAGE_ROOT / "requirements-data.txt"), "-e", str(PACKAGE_ROOT)], check=True)
os.chdir(PACKAGE_ROOT)
print({"package_revision": package_sha, "baseline_revision": baseline_sha})


## 2. Mount persistent storage

Check Drive free space for prepared data plus latest/best optimizer snapshots and the weights export.
Snapshots can be several times larger than weights. Raw ZIP/canonical intermediates live on ephemeral runtime disk.
Your Drive folder and all its evidence remain private unless you choose to share them.


In [ ]:
from google.colab import drive
import shutil
drive.mount("/content/drive")
PERSISTENT = Path("/content/drive/MyDrive/TRACE/teacher-clean-v1")
PERSISTENT.mkdir(parents=True, exist_ok=True)
RAW = Path("/content/unisar-data/raw")
CANONICAL = Path("/content/unisar-data/canonical")
PREPARED = PERSISTENT / "data/teacher-10000-v1"
RUNS = PERSISTENT / "runs"
EXPORT = PERSISTENT / "models/teacher/weights.pt"
CONFIG = PERSISTENT / "teacher-config.yaml"
if not CONFIG.exists():
    shutil.copy2(PACKAGE_ROOT / "configs/teacher-colab.yaml", CONFIG)
print({"persistent_folder": str(PERSISTENT), "runtime_free_GiB": round(shutil.disk_usage("/content").free / 2**30, 1)})


## 3. Download and verify the public dataset

No recruitment or scraping is needed. The downloader verifies published size/MD5 and records SHA-256.
Raw files use runtime disk. Skip downloading when persistent prepared data already exists and passes the later hash checks.


In [ ]:
def run_command(arguments):
    subprocess.run([str(x) for x in arguments], check=True, cwd=PACKAGE_ROOT)

if not (PREPARED / "manifest.json").exists():
    if shutil.disk_usage("/content").free < 15 * 2**30:
        raise RuntimeError("Reserve at least 15 GiB runtime disk for preparation.")
    run_command([sys.executable, "-m", "unisar.data.download", "--output", RAW])
else:
    print("Prepared data already exists; skip raw download and verify prepared hashes in preflight.")


## 4. Prepare the cohort and training-only vocabulary

The cohort uses training-only eligibility and deterministic nested ordering. Vocabulary fitting uses train only;
histories contain strictly earlier unordered timestamp blocks. Candidate groups are retained whole.
Preparation is CPU work; leave enough RAM and runtime disk. Existing manifests are immutable: reuse the folder or choose a new output root.


In [ ]:
if not (PREPARED / "manifest.json").exists():
    if not (CANONICAL / "10000/manifest.json").exists():
        run_command([sys.executable, "-m", "unisar.data.audit", "--archive", RAW / "KuaiSAR.zip",
                     "--output", CANONICAL, "--sizes", "10000"])
    run_command([sys.executable, "-m", "unisar.data.prepare", "--canonical", CANONICAL / "10000",
                 "--output", PREPARED])
else:
    print("Reusing persistent prepared dataset; preflight will verify its hashes.")


## 5. GPU/input/model/memory preflight — no optimizer updates

This checks actual device allocation, data contract, finite scores/loss/gradients and measured memory.
It also estimates parameter/Adam memory. It cannot guarantee memory for every rare slate or your remaining Colab quota.
Fix any failure before enabling training. Reducing batch size changes contrastive negatives and is a new configuration.


In [ ]:
run_command([sys.executable, "-m", "unisar.preflight", "--data", PREPARED, "--config", CONFIG,
             "--output", PERSISTENT / "preflight.json"])
import json
preflight = json.loads((PERSISTENT / "preflight.json").read_text())
print({k: preflight[k] for k in ("status", "device", "parameters", "fp32_weight_bytes", "gpu")})
for task, probe in preflight["probes"].items():
    print(task, {k: probe[k] for k in ("probe_requests", "probe_candidates", "largest_recorded_group")})


## 6. Start teacher training yourself

**Manual action:** change the gate below to `True` after reviewing preflight, then run this cell.
The starting budget is 10,000 paired updates, with a 50-minute cap per invocation; it is not a convergence claim.
Checkpoint I/O and full validation may extend that time. `latest.pt` is saved before validation and at regular boundaries.
After an interruption, rerun sections 1–5 and use the separate resume cell below.
Do not change planned steps, architecture, optimizer, AMP or dataset and call it an exact resume.


In [ ]:
RUN_TEACHER_TRAINING = False  # Change to True only when you want to start this run.
from unisar.training import load_config
teacher_config = load_config(CONFIG)
TEACHER_RUN = RUNS / teacher_config.run_id
if RUN_TEACHER_TRAINING:
    arguments = [sys.executable, "-m", "unisar.training", "train", "--data", PREPARED,
                 "--config", CONFIG, "--output-root", RUNS]
    if (TEACHER_RUN / "latest.pt").exists():
        raise RuntimeError("Checkpoint exists. Use section 7's resume gate to continue this run.")
    run_command(arguments)
else:
    print("Teacher training is disabled. Enable this gate and run the cell when ready.")


## 7. Persistent checkpoints and interrupted-session recovery

The trainer saves `latest.pt` to Drive at completed update boundaries, before validation and at a clean stop.
It includes optimizer, scheduler, sampler and RNG state; `best.pt` is selected by full validation.
After runtime loss, remount Drive and rerun setup/data/preflight. Leave the new-training gate false.
Enable this resume gate to continue the original schedule with the same config/data/implementation.
Exact CPU recovery is tested; GPU determinism and quota are not promised. A hard disconnect can lose work since the last atomic checkpoint.


In [ ]:
RUN_RESUME_TRAINING = False  # Enable only to continue an existing run.
if RUN_RESUME_TRAINING:
    if not (TEACHER_RUN / "latest.pt").exists():
        raise RuntimeError("No latest checkpoint found. Use section 6 to start a new run.")
    run_command([sys.executable, "-m", "unisar.training", "train", "--data", PREPARED,
                 "--config", CONFIG, "--output-root", RUNS, "--resume", "latest"])
else:
    print("Resume training is disabled. Latest checkpoint:", TEACHER_RUN / "latest.pt")


## 8. Inspect progress, export weights and collect validation outputs

Exports omit optimizer, RNG, sampler and internal training records. Keep the matching prepared vocabulary/configuration;
the export's data hash must match it. Selection uses full-validation equal-task supervised BCE, excluding final test.
A partial run may be exported for inspection, but quality/teacher viability are still pending.


In [ ]:
if (TEACHER_RUN / "metrics.json").exists():
    progress = json.loads((TEACHER_RUN / "metrics.json").read_text())
    print({k: progress[k] for k in ("status", "completed_steps", "planned_steps", "mean_call_step_seconds", "best_validation_joint_log_loss")})
if (TEACHER_RUN / "best.pt").exists():
    run_command([sys.executable, "-m", "unisar.training", "export", "--checkpoint", TEACHER_RUN / "best.pt", "--output", EXPORT])
    run_command([sys.executable, "-m", "unisar.training", "validate", "--data", PREPARED,
                 "--weights", EXPORT, "--device", "cuda", "--output", PERSISTENT / "validation"])
else:
    print("No best checkpoint yet. Run teacher training and full validation first.")


## 9. Optional longer matched baseline experiments

The existing short baselines do not establish convergence. Train one selected configuration per invocation on the same cohort.
Use shared/separate models and `full`, `no_current_query`, or `no_caption` policies to diagnose optimization and feature learning.
These are retrained controls; removing a channel from a frozen model answers a different question.
Keep planned steps/batch/padding/seed and tuning opportunities matched, and compare validation curves before conclusions.


In [ ]:
RUN_BASELINE_TRAINING = False  # Independent manual gate.
BASELINE_MODE = "shared"  # shared or separate
INPUT_POLICY = "full"  # full, no_current_query, no_caption
BASELINE_STEPS = 10000
if RUN_BASELINE_TRAINING:
    baseline_output = PERSISTENT / "baseline-runs"
    baseline_run_id = f"{BASELINE_MODE}-{INPUT_POLICY}-{BASELINE_STEPS}-v1"
    arguments = [sys.executable, PACKAGE_ROOT / "scripts/run_matched_baseline.py",
                 "--trace-root", TRACE_ROOT, "--data", PREPARED, "--output-root", baseline_output,
                 "--mode", BASELINE_MODE, "--input-policy", INPUT_POLICY,
                 "--steps", str(BASELINE_STEPS), "--batch-size", "32", "--max-seconds", "3000"]
    if (baseline_output / baseline_run_id / "latest.pt").exists():
        arguments += ["--resume", "latest"]
    run_command(arguments)
else:
    print("Matched baseline training is disabled. Select one configuration and enable its gate when ready.")


## Handoff and next decisions

Bring back the preflight report, source revisions, config, `metrics.json`, `validation-best.json` and exported weights.
Keep datasets, resume snapshots and dated evidence private. Next decisions depend on validation learning curves, query controls,
subgroup support and the measured teacher/compact-student quality-resource tradeoff. Final test and distillation are later gates.

**Quick check:** why does a 50-minute stop not imply completed training? Because the planned update budget and validation/convergence evidence
may still be incomplete; resumable sessions retain the original schedule. Inspect `completed_steps` versus `planned_steps` in the progress cell.

See the repository's `docs/colab.md`, `docs/paper-mapping.md` and `docs/data-contract.md` for the protocol and adaptation details.
